## A Character-Level Transformer for Machine Translation

**Author:** Diego Besada  
Created using [Pierre Nugues' template](https://github.com/pnugues/edan20/blob/master/labs_2026/17_02_translation_transformer_shared_embs.ipynb).

## Objectives

In this assignment, we build a character-level machine translation system using the encoder-decoder transformer architecture from _Attention Is All You Need_ [Vaswani et al., 2017](https://arxiv.org/abs/1706.03762).

The objectives are to:

* Know how machine translation works;
* Understand the principles of encoder-decoder architectures;
* Train this architecture on a parallel corpus;
* Compare results obtained with different configurations;
* Evaluate the models using sentence-level BLEU on an unseen test set.

## Overview

We implement a transformer model that translates between English and Spanish at the **character level**. The model shares a single vocabulary of 112 characters (including special tokens `<pad>`, `<unk>`, `<s>`, `</s>`) for both source and target languages.

The dataset is split into **train (80%)**, **validation (10%)**, and **test (10%)**. We use the Adam optimizer with a warmup learning rate schedule, dropout regularization, and early stopping based on validation loss.

The workflow supports two translation directions:
- **English -> Spanish**
- **Spanish -> English**

and two model sizes:
- **Large model:** `d_model=512, nhead=8, 3 encoder/decoder layers`
- **Small model:** `d_model=256, nhead=4, 2 encoder/decoder layers`

## Analysis

The transformer architecture [Vaswani et al., 2017](https://arxiv.org/abs/1706.03762) relies entirely on attention mechanisms, dispensing with recurrence and convolutions. The encoder maps an input sequence of symbol representations $(x_1, \ldots, x_n)$ to a continuous representation $\mathbf{z} = (z_1, \ldots, z_n)$. Given $\mathbf{z}$, the decoder then generates an output sequence $(y_1, \ldots, y_m)$ one element at a time.

The model uses **multi-head self-attention** and **positional encodings** to capture both content and position information. The feed-forward sublayers consist of two linear transformations with a ReLU activation in between.

For inference, we use **greedy decoding**: at each step, the model selects the most probable next character until the end-of-sequence token `</s>` is produced.

For evaluation, we use the **BLEU score** (Bilingual Evaluation Understudy) [Papineni et al., 2002](https://aclanthology.org/P02-1040/).

## Modules

In [14]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import LambdaLR

from torcheval.metrics.functional import bleu_score

import random
import numpy as np
import math

import matplotlib.pyplot as plt
from tqdm import tqdm
import json

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import warnings
warnings.filterwarnings('ignore')

## Hyperparameters

In [ ]:
# --- Translation direction ---
DIRECTION = 'es_en'   # 'en_es' (English -> Spanish) or 'es_en' (Spanish -> English)

# --- Model size ---
USE_MINI = False      # True: smaller model (faster); False: full model (professor's config)

# --- Device ---
DEVICE = torch.device('mps')

# --- Sequence length and vocabulary ---
MAX_LENGTH = 120 + 4          # 120 chars + <s> + </s> + <pad> + <unk>
SPECIAL_TOKENS = ['<pad>', '<unk>', '<s>', '</s>']
VOCAB_FILE = 'pico_translator.vocab'

# --- Data split ---
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
# Test ratio is the remaining 0.1

# --- Training ---
BATCH_SIZE = 64
EPOCHS = 50
PATIENCE = 7                  # Early stopping patience (epochs without val_loss improvement)
DROPOUT = 0.2                 # Dropout rate in Embedding and Transformer

# --- Optimizer and scheduler ---
WARMUP_STEPS = 4000

# --- Architecture (set by USE_MINI) ---
if USE_MINI:
    D_MODEL = 256
    N_HEAD = 4
    NUM_ENCODER_LAYERS = 2
    NUM_DECODER_LAYERS = 2
else:
    D_MODEL = 512
    N_HEAD = 8
    NUM_ENCODER_LAYERS = 3
    NUM_DECODER_LAYERS = 3

DIM_FEEDFORWARD = 4 * D_MODEL

# --- Derived ---
SRC_LANG = 'en' if DIRECTION == 'en_es' else 'es'
TGT_LANG = 'es' if DIRECTION == 'en_es' else 'en'
MODEL_NAME = f'translation_transformer_{DIRECTION}.pt'

## Dataset

We use the English-Spanish parallel corpus from [Tatoeba](http://www.manythings.org/anki/), stored in `spa.txt`. Each line contains a source sentence, a target sentence, and a license attribution, separated by tabs.

We extract **all** samples (`num_samples=None`) and keep only those where both source and target have length <= `MAX_LENGTH`.

In [16]:
def extract_pairs(file_path: str, max_len: int, num_samples=None):
    '''
    Extracts source and target sequences from a tab-separated file.
    Returns only pairs where both sequences have length <= max_len.
    If num_samples is None, all pairs are returned.
    '''
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
    random.shuffle(lines)
    src = []
    tgt = []
    for line in lines:
        src_seq, tgt_seq, _ = line.split('\t')
        if len(src_seq) <= max_len and len(tgt_seq) <= max_len:
            src.append(src_seq)
            tgt.append(tgt_seq)
    return src[:num_samples], tgt[:num_samples]

src, tgt = extract_pairs('spa.txt', MAX_LENGTH, num_samples=None)
print(f'Total pairs loaded: {len(src)}')
src[:3]
tgt[:3]

Total pairs loaded: 144109


['We have no more in stock.',
 'Tom used to drink like a fish.',
 'You should relax a bit.']

['No tenemos más existencias.',
 'Tom solía beber como pez.',
 'Debería relajarse un poco.']

## Train / Validation / Test Split

We split the dataset into:
- **Training set (80%)**: used for gradient updates;
- **Validation set (10%)**: used for early stopping and hyperparameter monitoring;
- **Test set (10%)**: used **only once** at the end to evaluate final BLEU score.

The split is applied identically to source and target to keep the pairs aligned.

In [17]:
n = len(src)
train_end = int(TRAIN_RATIO * n)
val_end = int((TRAIN_RATIO + VAL_RATIO) * n)

src_train, src_val, src_test = src[:train_end], src[train_end:val_end], src[val_end:]
tgt_train, tgt_val, tgt_test = tgt[:train_end], tgt[train_end:val_end], tgt[val_end:]

print(f'Train: {len(src_train)} | Val: {len(src_val)} | Test: {len(src_test)}')
print(f'Train ratio: {len(src_train)/n:.1%} | Val ratio: {len(src_val)/n:.1%} | Test ratio: {len(src_test)/n:.1%}')

Train: 115287 | Val: 14411 | Test: 14411
Train ratio: 80.0% | Val ratio: 10.0% | Test ratio: 10.0%


## Vocabulary

The vocabulary is **shared** between source and target. It is built from the union of all characters appearing in the training data, plus four special tokens:

| Token | Index | Description |
|-------|-------|-------------|
| `<pad>` | 0 | Padding token |
| `<unk>` | 1 | Unknown character |
| `<s>` | 2 | Start of sequence |
| `</s>` | 3 | End of sequence |

This results in 112 tokens (4 special + 108 characters).

In [18]:
charset = SPECIAL_TOKENS + sorted(list(set(''.join(src_train + tgt_train))))
idx2token = {i: token for i, token in enumerate(charset)}
token2idx = {token: i for i, token in enumerate(charset)}
print(f'Vocabulary size: {len(charset)}')
charset[:10]

Vocabulary size: 112


['<pad>', '<unk>', '<s>', '</s>', ' ', '!', '"', '$', '%', '&']

## Model Architecture

The model consists of two components:

1. **`Embedding`**: a character embedding layer with sinusoidal positional encoding and dropout;
2. **`Translator`**: a wrapper around `nn.Transformer` with weight sharing between the embedding layer and the output projection.

Both components use `dropout=0.2`.

In [19]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)

    def forward(self, x):
        out = self.embedding(x) * math.sqrt(self.d_model)
        out = out + self.pe[:x.size(1), :]
        return self.dropout(out)

In [20]:
class Translator(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, dim_feedforward, max_len, dropout=DROPOUT):
        super().__init__()
        self.embedding = Embedding(vocab_size, d_model, max_len, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(d_model, vocab_size)
        self.fc.weight = self.embedding.embedding.weight

    def forward(self, src, tgt, src_padding, tgt_padding):
        src_embs = self.embedding(src)
        tgt_embs = self.embedding(tgt)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(dim=1), device=tgt.device
        )
        x = self.transformer(
            src_embs,
            tgt_embs,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding,
            memory_key_padding_mask=src_padding,
            tgt_key_padding_mask=tgt_padding
        )
        return self.fc(x)

# --- Instantiate the model ---
if DIRECTION == 'es_en':
    _src_train, _tgt_train = tgt_train, src_train
    _src_val, _tgt_val = tgt_val, src_val
    _src_test, _tgt_test = tgt_test, src_test
else:
    _src_train, _tgt_train = src_train, tgt_train
    _src_val, _tgt_val = src_val, tgt_val
    _src_test, _tgt_test = src_test, tgt_test

translator = Translator(
    vocab_size=len(charset),
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    max_len=MAX_LENGTH + 2,
    dropout=DROPOUT
).to(DEVICE)

print(f'Model: {D_MODEL}d, {N_HEAD} heads, {NUM_ENCODER_LAYERS}/{NUM_DECODER_LAYERS} layers')
print(f'Dropout: {DROPOUT} | Parameters: {sum(p.numel() for p in translator.parameters()):,}')

Model: 512d, 8 heads, 3/3 layers
Dropout: 0.2 | Parameters: 22,128,752


## Optimizer and Learning Rate Schedule

We use the exact optimizer configuration from the professor:

$$\text{lr} = d_{\text{model}}^{-0.5} \cdot \min(\text{step}^{-0.5}, \text{step} \cdot \text{warmup}^{-1.5})$$

This is the warmup schedule from Vaswani et al. (2017), implemented via `LambdaLR`.

In [ ]:
def learning_rate(step_num, d_model, factor, warmup_steps):
    '''
    Computes the learning rate at a given step using the warmup schedule
    from Vaswani et al. (2017), 'Attention Is All You Need'.
    '''
    if step_num == 0:
        step_num = 1
    return factor * (
        d_model ** (-0.5) * min(step_num * warmup_steps ** (-1.5), step_num ** (-0.5))
    )

plt.figure(figsize=(8, 3))
plt.plot([learning_rate(i, D_MODEL, 1.0, WARMUP_STEPS) for i in range(0, 20000)])
plt.title('Learning Rate Schedule')
plt.xlabel('Step')
plt.ylabel('Learning Rate')
plt.grid(True)
plt.show()

for name, p in translator.named_parameters():
    if p.requires_grad and 'embedding' not in name and p.dim() > 1:
        nn.init.xavier_uniform_(p)

loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=0)

# --- Exact optimizer from the professor ---
optimizer = torch.optim.Adam(
    translator.parameters(), lr=1, betas=(0.9, 0.98), eps=1e-9
)

lr_scheduler = LambdaLR(
    optimizer=optimizer,
    lr_lambda=lambda step_num: learning_rate(
        step_num, d_model=D_MODEL, factor=1.0, warmup_steps=WARMUP_STEPS
    ),
)

## Data Pipeline

We define a `PairDataset` with a custom `collate_fn` that pads sequences within each batch to the same length. Three dataloaders are created: train, validation, and test.

In [21]:
def seqs2tensors(seqs, token2idx):
    '''Converts a list of strings to tensors of character indices.'''
    tensors = []
    for seq in seqs:
        seq = ['<s>'] + list(seq) + ['</s>']
        tensors += [torch.LongTensor([token2idx.get(x, 1) for x in seq])]
    return tensors

def tensors2seqs(tensors, idx2token):
    '''Converts tensors of indices back to lists of character lists.'''
    seqs = []
    for tensor in tensors:
        seqs += [[idx2token.get(x.item(), '<unk>') for x in tensor]]
    return seqs

class PairDataset(Dataset):
    def __init__(self, src, tgt, token2idx):
        self.src = src
        self.tgt = tgt
        self.token2idx = token2idx

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_batch = seqs2tensors([self.src[idx]], self.token2idx)
        tgt_batch = seqs2tensors([self.tgt[idx]], self.token2idx)
        return src_batch[0], tgt_batch[0]

    def collate_fn(self, batch):
        src_batch, tgt_batch = list(zip(*batch))
        src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0)
        tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
        return src_batch, tgt_batch

train_dataset = PairDataset(_src_train, _tgt_train, token2idx)
val_dataset = PairDataset(_src_val, _tgt_val, token2idx)
test_dataset = PairDataset(_src_test, _tgt_test, token2idx)

train_dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=train_dataset.collate_fn
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=val_dataset.collate_fn
)
test_dataloader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=test_dataset.collate_fn
)

## Training and Evaluation Functions

Both functions return per-character loss and accuracy, ignoring padding tokens.

In [ ]:
def train(model, loss_fn, optimizer, dataloader):
    model.train()
    t_loss = 0
    t_correct, t_chars = 0, 0
    for src_batch, tgt_batch in tqdm(dataloader):
        src_batch = src_batch.to(DEVICE)
        tgt_batch = tgt_batch.to(DEVICE)
        tgt_input = tgt_batch[:, :-1]
        tgt_output = tgt_batch[:, 1:]
        src_padding_mask = (src_batch == 0).to(DEVICE)
        tgt_padding_mask = (tgt_input == 0).to(DEVICE)
        tgt_output_pred = model(src_batch, tgt_input, src_padding_mask, tgt_padding_mask)
        optimizer.zero_grad()
        loss = loss_fn(
            tgt_output_pred.reshape(-1, tgt_output_pred.size(dim=-1)),
            tgt_output.reshape(-1)
        )
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        with torch.no_grad():
            n_chars = (tgt_output != 0).sum().item()
            t_chars += n_chars
            t_loss += loss.item() * n_chars
            char_pred = torch.argmax(tgt_output_pred, dim=-1)
            char_correct = torch.mul((char_pred == tgt_output), (tgt_output != 0)).sum().item()
            t_correct += char_correct
    return t_loss / t_chars, t_correct / t_chars

In [22]:
def evaluate(model, loss_fn, dataloader):
    model.eval()
    with torch.no_grad():
        t_loss = 0
        t_correct, t_chars = 0, 0
        for src_batch, tgt_batch in dataloader:
            src_batch = src_batch.to(DEVICE)
            tgt_batch = tgt_batch.to(DEVICE)
            tgt_input = tgt_batch[:, :-1]
            tgt_output = tgt_batch[:, 1:]
            src_padding_mask = (src_batch == 0)
            tgt_padding_mask = (tgt_input == 0)
            tgt_output_pred = model(src_batch, tgt_input, src_padding_mask, tgt_padding_mask)
            loss = loss_fn(
                tgt_output_pred.reshape(-1, tgt_output_pred.size(dim=-1)),
                tgt_output.reshape(-1)
            )
            n_chars = (tgt_output != 0).sum().item()
            t_chars += n_chars
            t_loss += loss.item() * n_chars
            char_pred = torch.argmax(tgt_output_pred, dim=-1)
            char_correct = torch.mul((char_pred == tgt_output), (tgt_output != 0)).sum().item()
            t_correct += char_correct
        return t_loss / t_chars, t_correct / t_chars

## Training Loop with Early Stopping

We train for up to `EPOCHS=50` epochs. After each epoch, we evaluate on the validation set. If `val_loss` does not improve for `PATIENCE=7` consecutive epochs, we restore the best weights and stop training.

In [ ]:
history = {
    'loss': [], 'val_loss': [],
    'accuracy': [], 'val_accuracy': []
}

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

for epoch in range(EPOCHS):
    train_loss, train_acc = train(translator, loss_fn, optimizer, train_dataloader)
    val_loss, val_acc = evaluate(translator, loss_fn, val_dataloader)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f"Epoch {epoch}: Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}, "
          f"Train acc: {train_acc:.3f}, Val acc: {val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.clone() for k, v in translator.state_dict().items()}
        torch.save(translator.state_dict(), MODEL_NAME)
        epochs_no_improve = 0
        print('  -> Model saved (improved)')
    else:
        epochs_no_improve += 1
        print(f'  -> No improvement ({epochs_no_improve}/{PATIENCE})')

    if epochs_no_improve >= PATIENCE:
        translator.load_state_dict(best_model_state)
        print(f'\nEarly stopping at epoch {epoch}. Restored best weights.')
        break

print(f'\nBest val loss: {best_val_loss:.4f}')

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, len(history['loss']) + 1), history['loss'], 'bo', label='Training loss')
ax1.plot(range(1, len(history['val_loss']) + 1), history['val_loss'], 'b', label='Validation loss')
ax1.set_title(f'Loss ({SRC_LANG} -> {TGT_LANG})')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, len(history['accuracy']) + 1), history['accuracy'], 'bo', label='Training acc')
ax2.plot(range(1, len(history['val_accuracy']) + 1), history['val_accuracy'], 'b', label='Validation acc')
ax2.set_title(f'Accuracy ({SRC_LANG} -> {TGT_LANG})')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Translation

We use greedy decoding for inference: at each step, the model predicts the most probable next character, which is fed back as input until `</s>` is generated.

In [23]:
def greedy_decode(model, src_tensor, max_len):
    '''
    Decodes token by token using the encoder's memory.
    Expects src_tensor with shape (1, src_len) or (src_len,).
    '''
    if src_tensor.dim() == 1:
        src_tensor = src_tensor.unsqueeze(0)
    src_tensor = src_tensor.to(DEVICE)
    src_embs = model.embedding(src_tensor)
    memory = model.transformer.encoder(src_embs)
    sos_idx = token2idx['<s>']
    tgt_idxs = torch.tensor([[sos_idx]], dtype=torch.long, device=DEVICE)
    for _ in range(max_len - 1):
        tgt_embs = model.embedding(tgt_idxs)
        tgt_len = tgt_idxs.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=DEVICE)
        decoder_out = model.transformer.decoder(tgt_embs, memory, tgt_mask=tgt_mask)
        last_token_logits = model.fc(decoder_out[:, -1, :])
        next_token_id = torch.argmax(last_token_logits, dim=-1, keepdim=True)
        tgt_idxs = torch.cat([tgt_idxs, next_token_id], dim=1)
        if next_token_id.item() == token2idx['</s>']:
            break
    return tgt_idxs[0, 1:]


def translate(model, src_sentence, extra_len=20):
    '''Translates a source sentence using greedy decoding.'''
    model.eval()
    with torch.no_grad():
        src_tokens = seqs2tensors([src_sentence.strip()], token2idx)[0]
        max_allowed_len = min(len(src_tokens) + extra_len, MAX_LENGTH)
        pred_indices = greedy_decode(model, src_tokens, max_len=max_allowed_len)
        pred_chars = tensors2seqs([pred_indices.cpu()], idx2token)[0]
        if pred_chars and pred_chars[-1] == '</s>':
            pred_chars = pred_chars[:-1]
        return ''.join(pred_chars)

## Translation Examples

In [24]:
translator.load_state_dict(torch.load(MODEL_NAME, map_location=DEVICE))
translator.eval()

if DIRECTION == 'en_es':
    examples = [
        'Come and eat.',
        'Where is the station?',
        'I like this book a lot.',
        'I do not have time.',
        'The weather is nice today.'
    ]
else:
    examples = [
        'Ven a comer.',
        'Donde esta la estacion?',
        'Me gusta este libro.',
        'No tengo tiempo.',
        'Hoy hace buen tiempo.'
    ]

for sent in examples:
    print(f'{sent}  ->  {translate(translator, sent)}')

<All keys matched successfully>

Translator(
  (embedding): Embedding(
    (dropout): Dropout(p=0.2, inplace=False)
    (embedding): Embedding(112, 512)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.2, inplace=False)
          (dropout2): Dropout(p=0.2, inplace=False)
        )
      )
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    )
    (d

Come and eat.  ->  Come un té.
Where is the station?  ->  ¿Dónde está el tren?
I like this book a lot.  ->  Me gusta este libro este libro.
I do not have time.  ->  No tengo tiempo para eso.
The weather is nice today.  ->  Ellos están decididos por el diccionario.


## BLEU Evaluation on Test Set

We evaluate the final model on the **held-out test set (10%)** using sentence-level BLEU score from `torcheval.metrics.functional.bleu_score`.

In [25]:
translator.load_state_dict(torch.load(MODEL_NAME, map_location=DEVICE))
translator.eval()
references = [seq.strip() for seq in _tgt_test]
candidates = [translate(translator, seq.strip()) for seq in tqdm(_src_test)]
test_bleu = bleu_score(candidates, references).item()
print(f'Test BLEU score ({SRC_LANG} -> {TGT_LANG}): {test_bleu:.4f}')
print(f'Test set size: {len(candidates)} sentences')

<All keys matched successfully>

Translator(
  (embedding): Embedding(
    (dropout): Dropout(p=0.2, inplace=False)
    (embedding): Embedding(112, 512)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.2, inplace=False)
          (dropout2): Dropout(p=0.2, inplace=False)
        )
      )
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    )
    (d

100%|██████████| 14411/14411 [1:09:15<00:00,  3.47it/s]  


Test BLEU score (en -> es): 0.0632
Test set size: 14411 sentences


## Save Model and Vocabulary

In [ ]:
torch.save(translator.state_dict(), MODEL_NAME)
with open(VOCAB_FILE, 'w') as f:
    json.dump(token2idx, f)
print(f'Model saved: {MODEL_NAME}')
print(f'Vocabulary saved: {VOCAB_FILE} ({len(charset)} tokens)')